In [1]:
# %%
!python3 -m spacy download en_core_web_trf

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 457.4/457.4 MB 25.9 MB/s eta 0:00:0000:0100:01
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_trf')


In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import spacy
import warnings
warnings.filterwarnings("ignore")
from spacy.tokens import DocBin
from tqdm import tqdm
from sklearn.model_selection import train_test_split

In [3]:
path = '../../data/augmented/'
df_train = pd.read_csv(path + 'aug_train_df.csv', encoding='utf-8', encoding_errors='ignore')
df_test = pd.read_csv(path + 'aug_test_df.csv', encoding='utf-8', encoding_errors='ignore')

In [4]:
df_train.head()

,Ingredient_Phrases,Labels
0,"3 ounces goat cheese , crumbled","['QUANTITY', 'UNIT', 'NAME', 'NAME', 'O', 'STA..."
1,"2 cups low sodium chicken broth , boiled until...","['QUANTITY', 'UNIT', 'STATE', 'STATE', 'NAME',..."
2,100 g minced beef,"['QUANTITY', 'UNIT', 'STATE', 'NAME']"
3,1 -2 cup shredded monterey jack cheese or 1 -2...,"['QUANTITY', 'QUANTITY', 'UNIT', 'STATE', 'NAM..."
4,"4 oranges , juice of -LRB- you can slip in som...","['QUANTITY', 'NAME', 'O', 'FORM', 'O', 'O', 'O..."


In [5]:
df_train.shape

(9549, 2)

In [6]:
# Step 1: Ensure all entries in 'Ingredient_Phrases' are strings
df_train["Ingredient_Phrases"] = df_train["Ingredient_Phrases"].astype(str)

# Step 2: Remove any empty or None values
df_train = df_train[df_train["Ingredient_Phrases"].str.strip() != '']  # Remove empty strings
df_train = df_train.dropna(subset=["Ingredient_Phrases"])  # Remove None or NaN values

In [7]:
print(df_train.dtypes)

Ingredient_Phrases    object
Labels                object
dtype: object


In [8]:
df_train.head()

,Ingredient_Phrases,Labels
0,"3 ounces goat cheese , crumbled","['QUANTITY', 'UNIT', 'NAME', 'NAME', 'O', 'STA..."
1,"2 cups low sodium chicken broth , boiled until...","['QUANTITY', 'UNIT', 'STATE', 'STATE', 'NAME',..."
2,100 g minced beef,"['QUANTITY', 'UNIT', 'STATE', 'NAME']"
3,1 -2 cup shredded monterey jack cheese or 1 -2...,"['QUANTITY', 'QUANTITY', 'UNIT', 'STATE', 'NAM..."
4,"4 oranges , juice of -LRB- you can slip in som...","['QUANTITY', 'NAME', 'O', 'FORM', 'O', 'O', 'O..."


In [9]:
import pandas as pd

def get_entity_tuple(df, label_map):
    list_of_entities = []
    
    for index, row in df.iterrows():
        ingredient_phrase = row["Ingredient_Phrases"]
        labels = row["Labels"]
        
        # Handle NaN or missing values
        if pd.isna(ingredient_phrase) or pd.isna(labels):
            continue  
        
        # Convert labels from string format to list format (if they are stored as strings)
        if isinstance(labels, str):
            labels = eval(labels)  # Use eval cautiously if data is safe
        
        # Tokenize by splitting on spaces
        words = ingredient_phrase.split()
        
        # Ensure the number of words matches the number of labels
        if len(words) != len(labels):
            print(f"Skipping row {index} due to mismatch in words ({len(words)}) and labels ({len(labels)})")
            continue
        
        # Extract entity spans
        entities = []
        start_idx = 0
        
        for word, label in zip(words, labels):
            end_idx = start_idx + len(word)
            mapped_label = label_map.get(label, "O")  # Default to 'O' if not in label_map
            
            if mapped_label != "O":  # Ignore 'O' labels
                entities.append((start_idx, end_idx, mapped_label))
            
            start_idx = end_idx + 1  # Account for space
        
        list_of_entities.append((ingredient_phrase, {"entities": entities}))  # Store the result
    
    return list_of_entities

# Example usage with label_map
label_map = {
    'QUANTITY': 'QUANTITY',
    'UNIT': 'UNIT',
    'NAME': 'NAME',
    'FORM': 'FORM',
    'STATE': 'STATE',
    'DF': 'DF',
    'SIZE': 'SIZE',
    'O': 'O'  # Outside token
}

# Call the function safely
train_entities = get_entity_tuple(df_train, label_map)
test_entities = get_entity_tuple(df_test, label_map)

Skipping row 7551 due to mismatch in words (7) and labels (6)
Skipping row 7558 due to mismatch in words (19) and labels (16)
Skipping row 7598 due to mismatch in words (20) and labels (19)
Skipping row 7601 due to mismatch in words (11) and labels (10)
Skipping row 7639 due to mismatch in words (18) and labels (16)
Skipping row 7643 due to mismatch in words (10) and labels (9)
Skipping row 7648 due to mismatch in words (14) and labels (13)
Skipping row 7655 due to mismatch in words (18) and labels (17)
Skipping row 7665 due to mismatch in words (16) and labels (14)
Skipping row 7675 due to mismatch in words (13) and labels (12)
Skipping row 7680 due to mismatch in words (11) and labels (9)
Skipping row 7704 due to mismatch in words (24) and labels (23)
Skipping row 7715 due to mismatch in words (15) and labels (14)
Skipping row 7730 due to mismatch in words (11) and labels (10)
Skipping row 7731 due to mismatch in words (11) and labels (9)
Skipping row 7739 due to mismatch in words (1

In [10]:
print(train_entities)

[('3 ounces goat cheese , crumbled', {'entities': [(0, 1, 'QUANTITY'), (2, 8, 'UNIT'), (9, 13, 'NAME'), (14, 20, 'NAME'), (23, 31, 'STATE')]}), ('2 cups low sodium chicken broth , boiled until reduced by half', {'entities': [(0, 1, 'QUANTITY'), (2, 6, 'UNIT'), (7, 10, 'STATE'), (11, 17, 'STATE'), (18, 25, 'NAME'), (26, 31, 'NAME'), (34, 40, 'STATE'), (47, 54, 'STATE'), (58, 62, 'FORM')]}), ('100 g minced beef', {'entities': [(0, 3, 'QUANTITY'), (4, 5, 'UNIT'), (6, 12, 'STATE'), (13, 17, 'NAME')]}), ('1 -2 cup shredded monterey jack cheese or 1 -2 cup colby , whatever you prefer', {'entities': [(0, 1, 'QUANTITY'), (2, 4, 'QUANTITY'), (5, 8, 'UNIT'), (9, 17, 'STATE'), (18, 26, 'NAME'), (27, 31, 'NAME'), (32, 38, 'NAME'), (42, 43, 'QUANTITY'), (44, 46, 'QUANTITY'), (47, 50, 'UNIT')]}), ('4 oranges , juice of -LRB- you can slip in some tangerine , too , if you like -RRB-', {'entities': [(0, 1, 'QUANTITY'), (2, 9, 'NAME'), (12, 17, 'FORM')]}), ('1 seedless cucumber , cut lengthwise and then

In [11]:
db = DocBin()
nlp = spacy.load("en_core_web_trf")

for text, annot in tqdm(train_entities):  # data in previous format
    doc = nlp.make_doc(text)  # create doc object from text
    ents = []
    for start, end, label in annot["entities"]:  # add character indexes
        span = doc.char_span(start, end, label=label, alignment_mode="contract")
        if span is None:
            print("Skipping entity")
        else:
            ents.append(span)
    doc.ents = ents  # label the text with the ents
    db.add(doc)
    
db.to_disk("../../data/spacy/train.spacy")

100%|██████████| 9402/9402 [00:00<00:00, 10704.57it/s]


In [12]:
# from tqdm import tqdm
# import spacy
# # from spacy.training import DocBin

# db = DocBin()
# nlp = spacy.load("en_core_web_trf")

# for text, annot in tqdm(train_entities):  # data in previous format
#     doc = nlp.make_doc(text)  # create doc object from text
#     ents = []
#     for start, end, label in annot["entities"]:  # add character indexes
#         # Change alignment_mode to "expand" for better handling of entities
#         span = doc.char_span(start, end, label=label, alignment_mode="expand")
#         if span is None:
#             print(f"Skipping entity: {text[start:end]}")  # Log problematic span
#         else:
#             ents.append(span)
#     doc.ents = ents  # label the text with the ents
#     db.add(doc)
    
# db.to_disk("../../data/spacy/train.spacy")


In [12]:
db = DocBin()
nlp = spacy.load("en_core_web_trf")

for text, annot in tqdm(test_entities):  # data in previous format
    doc = nlp.make_doc(text)  # create doc object from text
    ents = []
    for start, end, label in annot["entities"]:  # add character indexes
        span = doc.char_span(start, end, label=label, alignment_mode="contract")
        if span is None:
            print("Skipping entity")
        else:
            ents.append(span)
    doc.ents = ents  # label the text with the ents
    db.add(doc)

db.to_disk("../../data/spacy/test.spacy")  # save the docbin object

100%|██████████| 3235/3235 [00:00<00:00, 9658.75it/s]


In [14]:
# import os

# # Setting the CUDA and library paths
# os.environ['PATH'] = '/usr/local/cuda-11.6/bin:' + os.environ['PATH']
# os.environ['LD_LIBRARY_PATH'] = '/usr/local/cuda-11.6/lib64:' + os.environ.get('LD_LIBRARY_PATH', '')


In [13]:
!pip install spacy-transformers

In [16]:
import locale
def getpreferredencoding(do_setlocale=True):
    return "UTF-8"
locale.getpreferredencoding = getpreferredencoding
!python3 -m spacy train ../../configs/spacy/config.cfg --output ../../models/spacy_deberta_v3_large --paths.train ../../data/spacy/train.spacy --paths.dev ../../data/spacy/test.spacy --gpu-id 1

ℹ Saving to output directory: output
ℹ Using GPU: 1

=========================== Initializing pipeline ===========================
/home/ritisha21089/miniconda3/lib/python3.9/site-packages/thinc/shims/pytorch.py:261: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded fi

In [13]:
import locale
def getpreferredencoding(do_setlocale=True):
    return "UTF-8"
locale.getpreferredencoding = getpreferredencoding
!python3 -m spacy train ../../configs/spacy/config.cfg --output ../../models/spacy_deberta_v3_large --paths.train ../../data/spacy/train.spacy --paths.dev ../../data/spacy/test.spacy --gpu-id 1 


✔ Created output directory: output
ℹ Saving to output directory: output
ℹ Using GPU: 1

=========================== Initializing pipeline ===========================
/home/ritisha21089/miniconda3/lib/python3.9/site-packages/thinc/shims/pytorch.py:261: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't

In [17]:
import spacy
from spacy.training import Example
from sklearn.metrics import classification_report


# Load the trained model
nlp = spacy.load("../../models/spacy_deberta_v3_large/model-best")  # Adjust this path to your trained model's directory

def evaluate_model(nlp, test_entities, label_map):
    true_labels = []
    pred_labels = []

    # Map labels to integers for consistency with classification report
    label_to_id = {label: idx for idx, label in enumerate(label_map)}
    labels = list(label_to_id.values())  # List of label IDs for classification_report

    for text, annotation in test_entities:
        # Process the document with the model
        doc = nlp(text)
        example = Example.from_dict(doc, annotation)

        # Extract true entities from the example and predicted entities from the model
        true_ents = [(ent.start_char, ent.end_char, ent.label_) for ent in example.reference.ents]
        pred_ents = [(ent.start_char, ent.end_char, ent.label_) for ent in example.predicted.ents]

        # Token-wise alignment: generate token-level labels
        for token in doc:
            true_label = "O"  # Default to "O" if no entity
            pred_label = "O"  # Default to "O" if no entity

            # Check if the token is part of a true entity
            for start, end, label in true_ents:
                if token.idx >= start and token.idx < end:
                    true_label = label
                    break

            # Check if the token is part of a predicted entity
            for start, end, label in pred_ents:
                if token.idx >= start and token.idx < end:
                    pred_label = label
                    break

            # Convert labels to integer IDs
            true_labels.append(label_to_id.get(true_label, label_to_id["O"]))
            pred_labels.append(label_to_id.get(pred_label, label_to_id["O"]))

    # Generate classification report
    print("Classification Report:")
    print(classification_report(true_labels, pred_labels, labels=labels, target_names=label_map.keys(), zero_division=1))

# Define the label map, ensure it matches your data
label_map = {
    'QUANTITY': 'QUANTITY',
    'UNIT': 'UNIT',
    'NAME': 'NAME',
    'FORM': 'FORM',
    'STATE': 'STATE',
    'DF': 'DF',
    'SIZE': 'SIZE',
    'O': 'O'  # Outside token
}

# Run the evaluation function
evaluate_model(nlp, test_entities, label_map)

Classification Report:
              precision    recall  f1-score   support

    QUANTITY       0.99      0.99      0.99      3768
        UNIT       0.99      0.99      0.99      2544
        NAME       0.99      0.99      0.99      5224
        FORM       0.99      0.97      0.98      1160
       STATE       0.99      0.98      0.99      3123
          DF       0.99      0.98      0.98       328
        SIZE       0.95      0.97      0.96       678
           O       0.99      0.99      0.99      9512

    accuracy                           0.99     26337
   macro avg       0.99      0.98      0.98     26337
weighted avg       0.99      0.99      0.99     26337



In [18]:
import spacy
from spacy.training import Example
from sklearn.metrics import classification_report


# Load the trained model
nlp = spacy.load("../../models/spacy_deberta_v3_large/model-last")  # Adjust this path to your trained model's directory

def evaluate_model(nlp, test_entities, label_map):
    true_labels = []
    pred_labels = []

    # Map labels to integers for consistency with classification report
    label_to_id = {label: idx for idx, label in enumerate(label_map)}
    labels = list(label_to_id.values())  # List of label IDs for classification_report

    for text, annotation in test_entities:
        # Process the document with the model
        doc = nlp(text)
        example = Example.from_dict(doc, annotation)

        # Extract true entities from the example and predicted entities from the model
        true_ents = [(ent.start_char, ent.end_char, ent.label_) for ent in example.reference.ents]
        pred_ents = [(ent.start_char, ent.end_char, ent.label_) for ent in example.predicted.ents]

        # Token-wise alignment: generate token-level labels
        for token in doc:
            true_label = "O"  # Default to "O" if no entity
            pred_label = "O"  # Default to "O" if no entity

            # Check if the token is part of a true entity
            for start, end, label in true_ents:
                if token.idx >= start and token.idx < end:
                    true_label = label
                    break

            # Check if the token is part of a predicted entity
            for start, end, label in pred_ents:
                if token.idx >= start and token.idx < end:
                    pred_label = label
                    break

            # Convert labels to integer IDs
            true_labels.append(label_to_id.get(true_label, label_to_id["O"]))
            pred_labels.append(label_to_id.get(pred_label, label_to_id["O"]))

    # Generate classification report
    print("Classification Report:")
    print(classification_report(true_labels, pred_labels, labels=labels, target_names=label_map.keys(), zero_division=1))

# Define the label map, ensure it matches your data
label_map = {
    'QUANTITY': 'QUANTITY',
    'UNIT': 'UNIT',
    'NAME': 'NAME',
    'FORM': 'FORM',
    'STATE': 'STATE',
    'DF': 'DF',
    'SIZE': 'SIZE',
    'O': 'O'  # Outside token
}

# Run the evaluation function
evaluate_model(nlp, test_entities, label_map)

Classification Report:
              precision    recall  f1-score   support

    QUANTITY       0.98      0.98      0.98      2454
        UNIT       0.97      0.98      0.97      1710
        NAME       0.93      0.95      0.94      3464
        FORM       0.89      0.87      0.88       827
       STATE       0.94      0.94      0.94      2106
          DF       0.91      0.98      0.95       186
        SIZE       0.83      0.93      0.88       449
           O       0.95      0.93      0.94      6370

    accuracy                           0.95     17566
   macro avg       0.93      0.94      0.93     17566
weighted avg       0.95      0.95      0.95     17566

